## Create instructions for all data sets

In [28]:
STORY_INSTRUCTIONS = [
    "Tell me a story about <|insert|>",
    "Can you tell me a story about <|insert|>?",
    "Write a short story about <|insert|>",
    "Make up a story about <|insert|>",
    "I want to hear a story about <|insert|>",
    "Create a fun story about <|insert|>",
    "Please tell me a story about <|insert|>",
    "Invent a story about <|insert|>",
    "Tell a bedtime story about <|insert|>",
    "Can you make a story about <|insert|>?",
    
    "Wanna hear a fun story? Tell me about <|insert|>",
    "Hey, tell me a story about <|insert|>",
    "Could you write a story about <|insert|>?",
    "Give me a story about <|insert|>",
    "Do you know a story about <|insert|>?",
    
    "Tell me a simple story about <|insert|>",
    "Tell me a children's story about <|insert|>",
    "Tell me a happy story about <|insert|>",
    "Tell me a funny story about <|insert|>",
    "Tell me an interesting story about <|insert|>",
    
    "Write a creative story about <|insert|>",
    "Write a nice story about <|insert|>",
    "Write a fun little story about <|insert|>",
    "Write a short and simple story about <|insert|>",
    
    "Imagine a story about <|insert|> and tell it to me",
    "Can you imagine a story about <|insert|>?",
    "Make up a creative story about <|insert|>",
    
    "Let’s hear a story about <|insert|>",
    "Tell me something about <|insert|> in story form",
    "Turn <|insert|> into a story",
    
    "Tell me a story involving <|insert|>",
    "Create a story where <|insert|> is important",
    "Write a story that includes <|insert|>",
    
    "Can you tell me a bedtime story about <|insert|>?",
    "Tell me a relaxing story about <|insert|>",
    
    "Tell me a story with <|insert|> in it",
    "Write a story where <|insert|> appears",
]

WIKI_INSTRUCTIONS = [
    "Explain <|insert|> in simple terms.",
    "Write a short paragraph about <|insert|>.",
    "What is <|insert|>?",
    "Give an overview of <|insert|>.",
    "Describe how <|insert|> works.",
    "Why is <|insert|> important?",
    "Explain the concept of <|insert|>.",
    "Summarize <|insert|>.",
    "Teach me about <|insert|> like I'm a beginner.",
    "What are the key facts about <|insert|>?",
    "Write a simple explanation of <|insert|>.",
    "Explain <|insert|> with an example.",
]

WEB_INSTRUCTIONS = [
    "Explain <|insert|> in simple terms.",
    "Write a short paragraph about <|insert|>.",
    "What is <|insert|>?",
    "Describe <|insert|>.",
    "Give an overview of <|insert|>.",
    "Why is <|insert|> important?",
    "Explain the idea of <|insert|>.",
    "Summarize <|insert|>.",
    "Write about <|insert|>.",
    "Explain <|insert|> like I'm a beginner.",
]

In [14]:
import numpy as np

from NoTorchAI.Utils.Rake import Rake

MAX_BUFFER = 500_000
BUFFER = ''

In [15]:
def get_best_key_words_len_2_4(key_words: dict) -> str:
        key_words_2_to_4 = {
            k: v for k, v in key_words.items()
            if 2 <= len(k.split()) <= 4
        }
        if key_words_2_to_4:
            best_key, _ = max(key_words_2_to_4.items(), key=lambda kv: kv[1])
        else:
            best_key, _ = max(key_words.items(), key=lambda kv: kv[1])

        return best_key

In [ ]:

def stories_with_instructions():
    global BUFFER, MAX_BUFFER
    
    rake = Rake()
    rake.add_stop_word(["one", "day", "said", "hi", 
                        "now", "will", "says", "away", 
                        "always", "oh", "around"])

    with open("env/children_stories.txt", "r", encoding="utf-8") as f:
        stop = "<|endoftext|>"
        story = "" 

        for line_id, line in enumerate(f):
            if line_id == 5_000_000:
                break

            if stop in line:

                key_words = rake.get_key_words(story)
                if not key_words:
                    continue

                best_key = get_best_key_words_len_2_4(key_words)
                best_key = best_key.strip()
                if len(best_key.split()) > 6:
                    continue

                promt_template = np.random.choice(STORY_INSTRUCTIONS)  # returns scalar string
                promt = promt_template.replace("<|insert|>", best_key)
                BUFFER += "<|user|>: " + promt + "\n<|assistant|>: " + story + stop + "\n\n"
                story = ""
                continue

            if len(BUFFER) > MAX_BUFFER:
                with open("env/stories_instructions.txt", "a", encoding="utf-8") as out_file:
                    out_file.write(BUFFER)
                BUFFER = ""

            story += line

In [26]:
from datasets import load_dataset


def wiki_with_instructions():
    global BUFFER, MAX_BUFFER

    from datasets import load_dataset
    import numpy as np

    rake = Rake()
    rake.add_stop_word([
        "used", "using", "also", "known", "called",
        "one", "two", "first", "second",
        "many", "often", "usually"
])

    ds = load_dataset("rahular/simple-wikipedia", split="train")

    for row_id, row in enumerate(ds):
        if row_id == 5_000_000:
            break

        text = row.get("text") or row.get("document") or ""
        text = text.strip()

        if not text or len(text) < 200:  # Artificial length to exclude titels and so on
            continue

        # only first paragraph
        text = text.split("\n")[0]

        key_words = rake.get_key_words(text)
        if not key_words:
            continue

        best_key = get_best_key_words_len_2_4(key_words)
        best_key = best_key.strip()
        if len(best_key.split()) > 6:
            continue

        prompt_template = np.random.choice(WIKI_INSTRUCTIONS)
        prompt = prompt_template.replace("<|insert|>", best_key)

        BUFFER += "<|user|>: " + prompt + "\n"
        BUFFER += "<|assistant|>: " + text + "\n<|endoftext|>\n\n"

        if len(BUFFER) > MAX_BUFFER:
            with open("env/wiki_instructions.txt", "a", encoding="utf-8") as out_file:
                out_file.write(BUFFER)
            BUFFER = ""


In [ ]:
import itertools

from datasets import load_dataset


def web_with_instructions():
    global BUFFER, MAX_BUFFER

    import numpy as np

    rake = Rake()
    rake.add_stop_word([
        "said", "says", "also", "would", "could", "should",
        "one", "two", "first", "second",
        "many", "much", "very", "really",
        "get", "got", "go", "went",
        "like", "just", "even", "still",
        "know", "think", "people"
    ])

    ds = load_dataset("Skylion007/openwebtext", split="train", streaming=True)
    ds_small = itertools.islice(ds, 10000)

    for row_id, row in enumerate(ds):
        if row_id == 5_000_000:
            break

        text = row.get("text") or ""
        text = text.strip()

        if not text or len(text) < 200:
            continue

        if len(text) > 2000:
            text = text[:2000]

        if text.count("http") > 2:
            continue

        text = text.split("\n")[0]
        key_words = rake.get_key_words(text)
        if not key_words:
            continue

        best_key = get_best_key_words_len_2_4(key_words)
        best_key = best_key.strip()
        if len(best_key.split()) > 6:
            continue

        if any(char.isdigit() for char in best_key):
            continue

        if not best_key.lower().startswith(("a ", "an ", "the ")):
            best_key = "the " + best_key

        prompt_template = np.random.choice(WEB_INSTRUCTIONS)
        prompt = prompt_template.replace("<|insert|>", best_key)

        BUFFER += "<|user|>: " + prompt + "\n"
        BUFFER += "<|assistant|>: " + text + "\n<|endoftext|>\n\n"

        if len(BUFFER) > MAX_BUFFER:
            with open("env/web_instructions.txt", "a", encoding="utf-8") as out_file:
                out_file.write(BUFFER)
            BUFFER = ""

In [ ]:
BUFFER = ""
web_with_instructions()

## Hugging Face option

In [ ]:
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

ds1 = load_dataset("Skylion007/openwebtext")
ds2 = load_dataset("rahular/simple-wikipedia")


def all_texts():
    with open("children_stories.txt", "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield line.strip()

    for row in ds2:
        text = row.get("text") or row.get("document") or ""
        if text.strip():
            yield text

    for i, row in enumerate(ds1):
        if i >= 1_000_000:
            break
        if row["text"].strip():
            yield row["text"]


tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)  # ByteLevel handles punctuation/unicode better than Whitespace

trainer = trainers.BpeTrainer(
    vocab_size=16_000,
    min_frequency=2,
    special_tokens=["<|endofstory|>", "<|user|>", "<|assistant|>"]
)

tokenizer.train_from_iterator(all_texts(), trainer=trainer)  # train_from_iterator instead of train()

tokenizer.save("tokenizer.json")